# 00. Project Setup and Mathematical Conventions

This chapter defines the mathematical and computational conventions used throughout the project.

The goal of this project is not only to learn linear algebra as abstract matrix computation, but to understand how linear algebra supports the graphics pipeline:

$$
p_{\text{clip}}=
PVMp_{\text{model}}
$$

where:

- M is the model matrix;
- V is the view matrix;
- P is the projection matrix;
- p_model is a point in model space;
- p_clip is a point in clip space.

In this project, we will connect:

1. mathematical definitions;
2. concrete numerical examples;
3. NumPy implementations;
4. visualizations;
5. computer graphics applications.

我们约定采用列向量作为默认向量形式，采用矩阵在左、向量在右的布局形式，采用右手坐标系作为直角坐标系的具体形式。点和向量采用齐次坐标系的表示形式，其中点的特征坐标为1，向量的特征坐标为0

## 00.01 Project Goal

This project studies linear algebra for computer graphics.

Instead of following the traditional order of a pure linear algebra textbook, this project is organized around graphics problems:

- How do we represent points and directions?
- How do we move, rotate, and scale objects?
- How do we transform between coordinate systems?
- How do we project 3D points onto a 2D screen?
- How do we interpolate attributes inside a triangle?
- How do we compute lighting using vectors?
- How do we reason about numerical stability in graphics algorithms?

The central idea is:

> A matrix represents a transformation, and the graphics pipeline is a sequence of transformations.

## 00.02 Column Vector Convention

Throughout this project, we use the column vector convention.

A point or vector is represented as a column vector, and a transformation matrix acts on it from the left:

$$
p'=Mp
$$

When multiple transformations are composed, the rightmost transformation is applied first:

$$
p'=TRSp
$$
This means:

1. S is applied first;
2. R is applied second;
3. T is applied last.

This convention is common in mathematics and is convenient for explaining the graphics pipeline.

## 00.03 Matrix Multiplication Order

Matrix multiplication is generally not commutative:

$$
AB\ne BA
$$

In graphics, this means transformation order matters.

For example:

- scaling then rotating is not the same as rotating then scaling;
- rotating then translating is not the same as translating then rotating;
- local transformations and world transformations must be composed carefully.

Using the column vector convention:

$$
p'=TRp
$$

means the point p is first rotated by R, and then translated by T.

## 00.04 Right-Handed Coordinate System

This project uses a right-handed coordinate system.

In a right-handed 3D coordinate system:

- the x-axis points to the right;
- the y-axis points upward;
- the z-axis direction follows the right-hand rule.

The cross product follows the right-hand rule:

$$
x\times y=z
$$

This convention is useful for understanding:

- surface normals;
- camera coordinate frames;
- triangle orientation;
- back-face culling;
- rotation directions.

## 00.05 Points vs Vectors

In computer graphics, points and vectors have different geometric meanings.

A point represents a position in space.

A vector represents a direction, displacement, velocity, normal, or light direction.

A point can be moved by adding a vector:

$$
p'= p + v
$$

The difference between two points is a vector:

$$
v=p_2 - p_1
$$

However, adding two points directly usually has no clear geometric meaning.

This distinction will become especially important when we introduce homogeneous coordinates.

## 00.06 Homogeneous Coordinates

Homogeneous coordinates allow translation to be represented as matrix multiplication.

A point in 3D space is represented as:

$$
p=
\begin{bmatrix}
x \\
y \\
z \\
1
\end{bmatrix}
$$

A vector in 3D space is represented as:

$$
v=
\begin{bmatrix}
x \\
y \\
z \\
0
\end{bmatrix}
$$

The last coordinate is important:

- a point has w = 1;
- a vector has w = 0.

This means translation affects points but does not affect direction vectors.

## 00.07 NumPy Conventions

In this project, NumPy is used for concrete computation.

We use:

```python
import numpy as np
```

Vectors are usually represented as one-dimensional arrays:

```python
v = np.array([1.0, 2.0, 3.0])
```

Matrices are represented as two-dimensional arrays:

```python
M = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
])
```

Matrix-vector multiplication is written as:

```python
M @ v
```

Element-wise multiplication is written as:

```python
M * v
```

These two operations are not the same.

In [2]:
import numpy as np

# A 3D point
p = np.array([1.0, 2.0, 3.0])

# A displacement vector
v = np.array([4.0, 0.0, -1.0])

# Move the point by the vector
p_new = p + v

print("Original point:", p)
print("Displacement vector:", v)
print("New point:", p_new)

Original point: [1. 2. 3.]
Displacement vector: [ 4.  0. -1.]
New point: [5. 2. 2.]


In [3]:
## 补充称为齐次坐标

def to_homogeneous_point(p: np.ndarray) -> np.ndarray:
    """Convert a 3D point to homogeneous coordinates."""
    return np.append(p, 1.0)


def to_homogeneous_vector(v: np.ndarray) -> np.ndarray:
    """Convert a 3D vector to homogeneous coordinates."""
    return np.append(v, 0.0)


p = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 0.0, -1.0])

p_h = to_homogeneous_point(p)
v_h = to_homogeneous_vector(v)

print("Homogeneous point:", p_h)
print("Homogeneous vector:", v_h)

Homogeneous point: [1. 2. 3. 1.]
Homogeneous vector: [ 4.  0. -1.  0.]


In [ ]:
def translation_matrix(tx: float, ty: float, tz: float) -> np.ndarray:
    """Create a 4x4 translation matrix."""
    return np.array([
        [1.0, 0.0, 0.0, tx],
        [0.0, 1.0, 0.0, ty],
        [0.0, 0.0, 1.0, tz],
        [0.0, 0.0, 0.0, 1.0],
    ])


T = translation_matrix(10.0, 20.0, 30.0)

p = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 0.0, -1.0])

p_h = to_homogeneous_point(p)
v_h = to_homogeneous_vector(v)

p_transformed = T @ p_h
v_transformed = T @ v_h

## 可以看到这样设计的点和向量的表示系统具有的优点：平移变换改变点的位置，但是不改变方向的信息

print("Translation matrix:")
print(T)

print("\nTransformed point:")
print(p_transformed)

print("\nTransformed vector:")
print(v_transformed)

Translation matrix:
[[ 1.  0.  0. 10.]
 [ 0.  1.  0. 20.]
 [ 0.  0.  1. 30.]
 [ 0.  0.  0.  1.]]

Transformed point:
[11. 22. 33.  1.]

Transformed vector:
[ 4.  0. -1.  0.]


## 00.08 Exercises

### Exercise 1

Given:

$$
p=
\begin{bmatrix}
2 \\
-1 \\
4
\end{bmatrix}
,
\quad
v=
\begin{bmatrix}
3 \\
5 \\
-2
\end{bmatrix}
$$

Compute:

$$
p' = p + v
$$

Explain whether p' is a point or a vector.

---

### Exercise 2

Given two points:

$$
p_1 =
\begin{bmatrix}
1 \\
2 \\
3
\end{bmatrix}
,
\quad
p_2 =
\begin{bmatrix}
4 \\
6 \\
3
\end{bmatrix}
$$

Compute:

$$
v = p_2 - p_1
$$

Explain whether v is a point or a vector.

---

### Exercise 3

Write the homogeneous coordinates of the following point:

$$
p =
\begin{bmatrix}
3 \\
0 \\
-2
\end{bmatrix}
$$

---

### Exercise 4

Write the homogeneous coordinates of the following vector:

$$
v =
\begin{bmatrix}
3 \\
0 \\
-2
\end{bmatrix}
$$

---

### Exercise 5

Given the translation matrix:

$$
T =
\begin{bmatrix}
1 & 0 & 0 & 5 \\
0 & 1 & 0 & -2 \\
0 & 0 & 1 & 10 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

Apply T to the point:

$$
p =
\begin{bmatrix}
1 \\
1 \\
1 \\
1
\end{bmatrix}
$$

Then apply T to the vector:

$$
v =
\begin{bmatrix}
1 \\
1 \\
1 \\
0
\end{bmatrix}
$$

Explain why the results are different.